# `mutate` — Reference
`mutate` creates or overwrites columns using clean keyword argument syntax (`col=c.expr` or `col=callable`). Each entry is evaluated in order — a high-performance column expression per column, or a lambda when needed.
### Calling Styles
1. **Keyword arguments with `c.` expressions** (canonical & recommended): `.mutate(bmi=c.body_mass_g / c.bill_length_mm ** 2, mass_kg=c.body_mass_g / 1000)`
2. **Dictionary unpacking** (for new column names with spaces): `.mutate(**{"body mass kg": c.body_mass_g / 1000})`
3. **External file specification**: `.mutate("@features.txt")`
---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src"))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), "..", "src"))
if _src not in sys.path: sys.path.insert(0, _src)

import numpy as np
import pytae as pt
from pytae import c, if_else, case_when

penguins = pt.sample_data["penguins"]
tips = pt.sample_data["tips"]
titanic = pt.sample_data["titanic"]


## A single derived column

In [2]:
# Body mass index style ratio — clean assignment formula via canonical c kwargs
(
    penguins
    .mutate(bmi=c.body_mass_g / (c.bill_length_mm ** 2))
    .select(c.species, c.body_mass_g, c.bill_length_mm, c.bmi)
    .sample(10)
)


species,body_mass_g,bill_length_mm,bmi
str,f64,f64,f64
"""Adelie""",3600.0,38.9,2.379049
"""Adelie""",4075.0,42.7,2.234971
"""Adelie""",3325.0,38.5,2.243211
"""Adelie""",3500.0,43.1,1.884141
"""Chinstrap""",3900.0,47.5,1.728532
"""Gentoo""",5150.0,46.8,2.35134
"""Gentoo""",5050.0,45.1,2.48278
"""Gentoo""",4200.0,45.3,2.046694
"""Gentoo""",4625.0,49.1,1.918442


In [3]:
# Legacy string formulas are retired and raise TypeError immediately
try:
    penguins.mutate(bmi="body_mass_g / bill_length_mm ** 2")
except TypeError as exc:
    print(f"TypeError (as expected): {exc}")


TypeError (as expected): String formula 'body_mass_g / bill_length_mm ** 2' in mutate() is retired. Use canonical c expressions, e.g. df.mutate(bmi=c...)


## Multiple entries in one call, and chaining a later entry off an earlier one
Entries are applied left to right, so a later expression can reference a column derived earlier in the *same* `mutate()` call.

In [4]:
# Two independent derived columns in one call via kwargs
(
    penguins
    .mutate(heavy=c.body_mass_g > 4000, mass_kg=c.body_mass_g / 1000)
    .select(c.species, c.body_mass_g, c.mass_kg, c.heavy)
    .sample(10)
)


species,body_mass_g,mass_kg,heavy
str,f64,f64,bool
"""Adelie""",4250.0,4.25,true
"""Adelie""",3800.0,3.8,false
"""Adelie""",4275.0,4.275,true
"""Adelie""",3050.0,3.05,false
"""Adelie""",4000.0,4.0,false
"""Chinstrap""",4450.0,4.45,true
"""Gentoo""",4100.0,4.1,true
"""Gentoo""",4400.0,4.4,true
"""Gentoo""",4950.0,4.95,true


In [5]:
# mass_lb references mass_kg, derived by the keyword just before it
(
    penguins
    .mutate(mass_kg=c.body_mass_g / 1000, mass_lb=c.mass_kg * 2.20462)
    .select(c.species, c.body_mass_g, c.mass_kg, c.mass_lb)
    .sample(10)
)


species,body_mass_g,mass_kg,mass_lb
str,f64,f64,f64
"""Adelie""",3900.0,3.9,8.598018
"""Adelie""",3600.0,3.6,7.936632
"""Adelie""",3550.0,3.55,7.826401
"""Adelie""",4300.0,4.3,9.479866
"""Adelie""",3975.0,3.975,8.7633645
"""Chinstrap""",3450.0,3.45,7.605939
"""Chinstrap""",3800.0,3.8,8.377556
"""Gentoo""",5150.0,5.15,11.353793
"""Gentoo""",5800.0,5.8,12.786796


## String comparisons and local variables
String literals *inside* the expression (e.g. `'Adelie'`) need real quotes — column names stay bare. A variable from the calling scope can be referenced with an `@` prefix, same as pandas' own `eval()`/`query()`.


In [6]:
# Boolean comparison expressions
(
    penguins
    .mutate(is_adelie=c.species == "Adelie")
    .select(c.species, c.is_adelie)
    .sample(10)
)


species,is_adelie
str,bool
"""Adelie""",true
"""Adelie""",true
"""Adelie""",true
"""Adelie""",true
"""Adelie""",true
"""Adelie""",true
"""Chinstrap""",false
"""Gentoo""",false
"""Gentoo""",false


In [7]:
# Using external scalar variables directly with c expressions
threshold = 4000

(
    penguins
    .mutate(heavy=c.body_mass_g >= threshold)
    .select(c.species, c.body_mass_g, c.heavy)
    .sample(10)
)


species,body_mass_g,heavy
str,f64,bool
"""Adelie""",3325.0,false
"""Adelie""",4150.0,true
"""Adelie""",3775.0,false
"""Adelie""",4300.0,true
"""Adelie""",3425.0,false
"""Chinstrap""",3525.0,false
"""Chinstrap""",3950.0,false
"""Gentoo""",4650.0,true
"""Gentoo""",3950.0,false


## Overwriting an existing column

In [8]:
# mutate() can overwrite a column in place, e.g. converting units
(
    penguins
    .mutate(body_mass_g=c.body_mass_g / 1000)
    .select(c.species, c.body_mass_g)
    .sample(10)
)


species,body_mass_g
str,f64
"""Adelie""",3.9
"""Adelie""",2.975
"""Adelie""",3.475
"""Chinstrap""",4.15
"""Gentoo""",4.15
"""Gentoo""",5.3
"""Gentoo""",4.75
"""Gentoo""",4.85
"""Gentoo""",5.0


## Column names with spaces — bracket syntax
Use square brackets `[col name]` to reference column names containing spaces without shell or syntax hazards (backticks are also supported):

In [9]:
import pandas as pd
spaced = pt.DataFrame({"body mass g": [3750, 4200], "bill length mm": [39.1, 46.5]})

# c["..."] protects column names that contain spaces
spaced.mutate(bmi=c["body mass g"] / (c["bill length mm"] ** 2))


body mass g,bill length mm,bmi
i64,f64,f64
3750,39.1,2.452888
4200,46.5,1.942421


## Real-world pipeline: mutate → filter → select
`mutate()` chains like any other pytae method — filter on a column you just derived with `filter()`.

In [10]:
(
    penguins
    .mutate(bmi=c.body_mass_g / (c.bill_length_mm ** 2))
    .filter(c.bmi > 2)
    .select(c.species, c.island, c.body_mass_g, c.bill_length_mm, c.bmi)
    .sample(10)
)


species,island,body_mass_g,bill_length_mm,bmi
str,str,f64,f64,f64
"""Adelie""","""Torgersen""",3450.0,36.7,2.561456
"""Adelie""","""Torgersen""",3300.0,37.8,2.309566
"""Adelie""","""Torgersen""",3800.0,36.7,2.821314
"""Adelie""","""Dream""",3350.0,37.3,2.407837
"""Adelie""","""Dream""",4450.0,39.6,2.837721
"""Adelie""","""Biscoe""",4775.0,43.2,2.558621
"""Adelie""","""Torgersen""",3875.0,41.4,2.260846
"""Gentoo""","""Biscoe""",4100.0,44.5,2.070446
"""Gentoo""","""Biscoe""",4600.0,47.5,2.038781


## Conditional column creation — `if_else()`, `case_when()`, `map()`
`mutate()` provides vectorized conditional branching helpers:
- `if_else(condition, true_value, false_value)`: Binary branching between literals, expressions, or columns.
- `case_when((cond1, val1), (cond2, val2), ..., default=val)`: Ordered multi-tier priority branching (like SQL CASE WHEN / dplyr case_when).
- `col.map({key: value, ...}, default=val)`: Category recoding via dictionary lookup.


In [11]:
# if_else(condition, true_value, false_value) — binary branching
(
    penguins
    .mutate(weight_class=if_else(c.body_mass_g > 4000, "heavy", "light"))
    .select(c.species, c.body_mass_g, c.weight_class)
    .sample(10)
)


species,body_mass_g,weight_class
str,f64,str
"""Adelie""",3800.0,"""light"""
"""Adelie""",3325.0,"""light"""
"""Adelie""",4650.0,"""heavy"""
"""Adelie""",4725.0,"""heavy"""
"""Adelie""",3275.0,"""light"""
"""Adelie""",3975.0,"""light"""
"""Chinstrap""",3800.0,"""light"""
"""Gentoo""",5000.0,"""heavy"""
"""Gentoo""",4850.0,"""heavy"""


In [12]:
# Multi-tier branching with case_when()
(
    penguins
    .mutate(
        size_class=case_when(
            (c.body_mass_g >= 4500, "large"),
            (c.body_mass_g >= 3500, "medium"),
            default="small",
        )
    )
    .select(c.species, c.body_mass_g, c.size_class)
    .sample(10)
)


species,body_mass_g,size_class
str,f64,str
"""Adelie""",4200.0,"""medium"""
"""Adelie""",3400.0,"""small"""
"""Adelie""",3950.0,"""medium"""
"""Adelie""",3300.0,"""small"""
"""Adelie""",4000.0,"""medium"""
"""Chinstrap""",3775.0,"""medium"""
"""Chinstrap""",4500.0,"""large"""
"""Chinstrap""",3325.0,"""small"""
"""Gentoo""",4700.0,"""large"""


In [13]:
# map(dict, default=...) — recode a column through a dictionary lookup
(
    penguins
    .mutate(
        island_code=c.island.map({"Torgersen": "TOR", "Biscoe": "BIS"}, default="OTH")
    )
    .select(c.species, c.island, c.island_code)
    .sample(10)
)


species,island,island_code
str,str,str
"""Adelie""","""Biscoe""","""BIS"""
"""Adelie""","""Biscoe""","""BIS"""
"""Adelie""","""Dream""","""OTH"""
"""Adelie""","""Dream""","""OTH"""
"""Chinstrap""","""Dream""","""OTH"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""
"""Gentoo""","""Biscoe""","""BIS"""


## First non-null resolution — `coalesce()`

`coalesce(col1, col2, ..., default)` evaluates candidates left-to-right and returns the first non-null value per row, like SQL `COALESCE()` or `dplyr::coalesce()`:

In [14]:
contacts = pt.DataFrame({
    "mobile": [None, "555-1234", None],
    "home": ["555-5678", None, None],
    "work": [None, None, "555-9012"],
})

contacts.mutate(preferred_contact=c.mobile.coalesce(c.home, c.work, "N/A"))


mobile,home,work,preferred_contact
str,str,str,str
null,"""555-5678""",null,"""555-5678"""
"""555-1234""",null,null,"""555-1234"""
null,null,"""555-9012""","""555-9012"""


## Combining expressions and callables in kwargs

`mutate()` cleanly mixes expression strings and python callables (e.g. lambdas) in keyword arguments:


In [15]:
(
    penguins
    .mutate(
        bmi=c.body_mass_g / (c.bill_length_mm ** 2),
        mass_kg=c.body_mass_g / 1000,
        is_heavy=lambda df: df["body_mass_g"] > 4000,
    )
    .select(c.species, c.bmi, c.mass_kg, c.is_heavy)
    .head(5)
)


species,bmi,mass_kg,is_heavy
str,f64,f64,bool
"""Adelie""",2.452888,3.75,false
"""Adelie""",2.435507,3.8,false
"""Adelie""",2.001121,3.25,false
"""Adelie""",null,null,null
"""Adelie""",2.561456,3.45,false


### Row-wise custom logic with Python's built-in `map()`
When custom logic requires multi-column `if/elif/else` branching or arbitrary Python functions expecting row values, wrap the scalar function in Python's built-in `map()` inside a callable `lambda d: map(func, d['a'], d['b'])`.

For dictionary value recoding, use vectorized formula `map(col, mapping)` instead.

In [16]:
def classify_penguin(species, mass_kg, island):
    if mass_kg is None or np.isnan(mass_kg):
        return "Incomplete Data"
    if species == "Gentoo" and mass_kg > 5.0:
        return "Giant Gentoo"
    elif island == "Torgersen" and mass_kg > 3.7:
        return "Prime Torgersen"
    else:
        return "Standard"

(
    penguins
    .mutate(
        mass_kg=c.body_mass_g / 1000,
        category=lambda d: map(classify_penguin, d["species"], d["mass_kg"], d["island"]),
    )
    .select(c.species, c.island, c.body_mass_g, c.mass_kg, c.category)
    .head(6)
)


species,island,body_mass_g,mass_kg,category
str,str,f64,f64,str
"""Adelie""","""Torgersen""",3750.0,3.75,"""Prime Torgersen"""
"""Adelie""","""Torgersen""",3800.0,3.8,"""Prime Torgersen"""
"""Adelie""","""Torgersen""",3250.0,3.25,"""Standard"""
"""Adelie""","""Torgersen""",null,null,"""Incomplete Data"""
"""Adelie""","""Torgersen""",3450.0,3.45,"""Standard"""
"""Adelie""","""Torgersen""",3650.0,3.65,"""Standard"""


## Loading specs from an external file — `@specs.txt`
For complex feature engineering or shared pipelines across Python and the CLI, specs can be loaded from an external file. The file supports multiline expressions, `#` comments, empty lines, and column chaining (where later formulas reference columns defined earlier in the same file).
### File content of `penguin_features.txt`:
```text
# Engineering features for penguins
mass_kg = body_mass_g / 1000
# Convert kg to lbs (references mass_kg computed above)
mass_lb = mass_kg * 2.20462
```


In [17]:
# Feature engineering using canonical c expressions
result = penguins.mutate(
    mass_kg=c.body_mass_g / 1000,
    mass_lb=c.body_mass_g / 1000 * 2.20462
)
out = (
    result
    .select(c.species, c.mass_kg, c.mass_lb)
    .sample(5)
)
out


species,mass_kg,mass_lb
str,f64,f64
"""Adelie""",3.75,8.267325
"""Gentoo""",4.3,9.479866
"""Gentoo""",4.6,10.141252
"""Gentoo""",5.95,13.117489
"""Gentoo""",4.625,10.196367


## Columns with spaces — SQL-style brackets `[col]`

SQL-style square brackets `[col a]` are supported alongside backticks, avoiding shell backtick substitution hazards:

In [18]:
spaced.mutate(ratio=c["body mass g"] / c["bill length mm"])


body mass g,bill length mm,ratio
i64,f64,f64
3750,39.1,95.907928
4200,46.5,90.322581


## Creating new columns with spaces — string expressions `[col] = ...`

You can create columns with spaces directly using string assignment expressions with brackets `[new col] = ...` with zero dictionary unpacking.

(Dictionary unpacking `**{'col name': ...}` is also supported for kwargs).

In [19]:
# Create a new column with spaces using dict unpacking
(
    spaced
    .mutate(**{"body mass ratio": c["body mass g"] / c["bill length mm"]})
    .select(c["body mass g"], c["body mass ratio"])
    .head()
)


body mass g,body mass ratio
i64,f64
3750,95.907928
4200,90.322581


In [20]:
# Define multiple new columns with spaces via dict unpacking
(
    penguins
    .mutate(**{
        "body mass kg": c.body_mass_g / 1000,
        "bill ratio": c.bill_length_mm / c.bill_depth_mm,
    })
    .select(c.species, c["body mass kg"], c["bill ratio"])
    .head()
)


species,body mass kg,bill ratio
str,f64,f64
"""Adelie""",3.75,2.090909
"""Adelie""",3.8,2.270115
"""Adelie""",3.25,2.238889
"""Adelie""",null,null
"""Adelie""",3.45,1.901554


Note: When loading specs from an external file (`@specs.txt`) or via the CLI (`-mutate`), names with spaces can be written directly on the left-hand side of `=` without dictionary unpacking:

```text
# inside an external spec file or CLI flag:
my bmi = [body mass g] / [bill length mm] ** 2
```


## Grouped mutations — `df.by(...).mutate(...)`

Unlike `pt.agg()` which collapses rows ($N \to K$), `df.by(...).mutate(...)` evaluates window calculations and group-level summaries and broadcasts them back to every row ($N \to N$).

Supported features:
- **Injected aggregation functions**: `mean(x)`, `sum(x)`, `median(x)`, `min(x)`, `max(x)`, `std(x)`, `var(x)`
- **Group row count**: `count()` (canonical window counting function)
- **Sequential expressions**: Later expressions within the same `mutate` call can immediately reference earlier group-derived columns (e.g. `diff = tip - avg_tip`)
- **Arbitrary output column names**: Name target columns cleanly without hardcoded prefixes
- **Multi-column grouping**: `.by('species', 'island')`
- **Spaced column names**: `[total bill]`
- **Missing group handling**: `dropna=False` retains NA groups


In [21]:
# Compute group average tip, deviation from group average, and group sample size
tips = pt.sample("tips")
(
    tips
    .select(c.day, c.time, c.total_bill, c.tip)
    .by("day")
    .mutate(avg_tip=c.tip.mean(), diff=c.tip - c.tip.mean(), group_size=c.tip.count())
    .head(6)
)


day,time,total_bill,tip,avg_tip,diff,group_size
cat,cat,f64,f64,f64,f64,u32
"""Sun""","""Dinner""",16.99,1.01,3.255132,-2.245132,76
"""Sun""","""Dinner""",10.34,1.66,3.255132,-1.595132,76
"""Sun""","""Dinner""",21.01,3.5,3.255132,0.244868,76
"""Sun""","""Dinner""",23.68,3.31,3.255132,0.054868,76
"""Sun""","""Dinner""",24.59,3.61,3.255132,0.354868,76
"""Sun""","""Dinner""",25.29,4.71,3.255132,1.454868,76


In [22]:
# Multi-column grouping with spaced columns
(
    tips
    .rename(columns={"total_bill": "total bill"})
    .select(c.day, c.time, c["total bill"], c.tip)
    .by("day", "time")
    .mutate(**{"avg bill": c["total bill"].mean(), "bill diff": c["total bill"] - c["total bill"].mean(), "count": c.tip.count()})
    .head(6)
)


day,time,total bill,tip,avg bill,bill diff,count
cat,cat,f64,f64,f64,f64,u32
"""Sun""","""Dinner""",16.99,1.01,21.41,-4.42,76
"""Sun""","""Dinner""",10.34,1.66,21.41,-11.07,76
"""Sun""","""Dinner""",21.01,3.5,21.41,-0.4,76
"""Sun""","""Dinner""",23.68,3.31,21.41,2.27,76
"""Sun""","""Dinner""",24.59,3.61,21.41,3.18,76
"""Sun""","""Dinner""",25.29,4.71,21.41,3.88,76


In [23]:
# De-meaning and z-score calculation per group
(
    penguins
    .select(c.species, c.body_mass_g)
    .by("species")
    .mutate(
        mean_mass=c.body_mass_g.mean(),
        std_mass=c.body_mass_g.std(),
        z_score=(c.body_mass_g - c.body_mass_g.mean()) / c.body_mass_g.std(),
    )
    .head(6)
)


species,body_mass_g,mean_mass,std_mass,z_score
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,458.566126,0.107591
"""Adelie""",3800.0,3700.662252,458.566126,0.216627
"""Adelie""",3250.0,3700.662252,458.566126,-0.982764
"""Adelie""",null,3700.662252,458.566126,null
"""Adelie""",3450.0,3700.662252,458.566126,-0.546622
"""Adelie""",3650.0,3700.662252,458.566126,-0.11048


### Grouped vs. Overall Window Calculations (Single-Shot Grouping)

In Pytae, `.by()` establishes a single-shot, ephemeral grouping context consumed immediately by the next operation. Once `mutate()` finishes, the returned DataFrame is automatically flat and ungrouped! You can freely chain from a grouped mutation directly into an overall calculation without ever needing `ungroup()`.

In [24]:
# Comparing species-level mean to overall dataset mean in a single pipeline
(
    penguins
    .select(c.species, c.body_mass_g)
    .by("species")
    .mutate(species_mean=c.body_mass_g.mean())
    .mutate(overall_mean=c.body_mass_g.mean())
    .mutate(species_vs_overall=c.species_mean - c.overall_mean)
    .head(6)
)


species,body_mass_g,species_mean,overall_mean,species_vs_overall
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3800.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3250.0,3700.662252,4201.754386,-501.092134
"""Adelie""",null,3700.662252,4201.754386,-501.092134
"""Adelie""",3450.0,3700.662252,4201.754386,-501.092134
"""Adelie""",3650.0,3700.662252,4201.754386,-501.092134


### Handling missing grouping values with `dropna`

By default, `dropna=False` computes group statistics across `NA` rows as their own distinct group (no silent data loss). Pass `dropna=True` to exclude rows with `NA` in grouping columns from calculations, filling `NaN` in mutated columns while preserving total row count and index alignment:


In [25]:
sample_data = pt.DataFrame({
    "grp": ["A", "A", None, "B", None],
    "val": [10.0, 20.0, 100.0, 30.0, 200.0]
})

print("Grouped mutate:")
print(
    sample_data
    .by("grp")
    .mutate(avg=c.val.mean(), sz=c.val.count())
)


Grouped mutate:
shape: (5, 4)
┌──────┬───────┬───────┬─────┐
│ grp  ┆ val   ┆ avg   ┆ sz  │
│ ---  ┆ ---   ┆ ---   ┆ --- │
│ str  ┆ f64   ┆ f64   ┆ u32 │
╞══════╪═══════╪═══════╪═════╡
│ A    ┆ 10.0  ┆ 15.0  ┆ 2   │
│ A    ┆ 20.0  ┆ 15.0  ┆ 2   │
│ null ┆ 100.0 ┆ 150.0 ┆ 2   │
│ B    ┆ 30.0  ┆ 30.0  ┆ 1   │
│ null ┆ 200.0 ┆ 150.0 ┆ 2   │
└──────┴───────┴───────┴─────┘


## Comprehensive end-to-end recipe: combining all `mutate` capabilities

This master example combines all core features in a single fluent pipeline:
1. **Normal mutate**: Standard arithmetic derivation (`bill_ratio`, `mass_kg`).
2. **Sequential column referencing**: Using a freshly created column (`mass_kg`) inside subsequent expressions (`bmi`, `above_target`, `size_label`) within the *same* `mutate()` call.
3. **Binary conditional (`if_else`)**: Conditional branching into `'Heavy'` / `'Standard'`.
4. **Multi-condition tiers (`case_when`)**: Multi-tier categorization with ordered tuples and a fallback `default`.
5. **Null fallback (`coalesce`)**: Resolving the first non-null value across columns or supplying a fallback for missing data.
6. **Dictionary mapping (`map`)**: Category recoding through a dictionary lookup with fallback handling.
7. **Local variables (`@var`)**: Referencing caller-scope Python variables directly using the `@` prefix.
8. **Columns with spaces (`[col]`)**: Defining spaced columns via dict unpacking and referencing them with brackets `[mass lbs]`.
9. **Grouped window transforms (`pt.by()`)**: Computing group-level aggregations (`mean(mass_kg)`), relative offsets (`mass_diff`), and group sizes (`n`) without collapsing rows.
10. **Lambda callables**: Custom function inspecting the accumulated DataFrame with all prior mutations already attached.


In [26]:
# Complex feature engineering pipeline using canonical c expressions
master_df = (
    penguins
    .mutate(
        mass_kg=c.body_mass_g / 1000.0,
        bill_ratio=c.bill_length_mm / c.bill_depth_mm,
        bmi=c.body_mass_g / (c.bill_length_mm ** 2),
        above_target=c.body_mass_g >= 4000,
        size_label=if_else(c.body_mass_g >= 4200, "Heavy", "Normal"),
        effective_sex=c.sex.coalesce("Undetermined"),
        species_code=c.species.map({"Adelie": "AD", "Gentoo": "GE", "Chinstrap": "CH"}, default="OTHER"),
        **{"mass lbs": c.mass_kg * 2.20462},
    )
)
cols_to_show = [c.species, c.species_code, c.mass_kg, c.bmi, c.size_label, c.effective_sex, c.above_target]
master_df.select(*cols_to_show).head(8)


species,species_code,mass_kg,bmi,size_label,effective_sex,above_target
str,str,f64,f64,str,str,bool
"""Adelie""","""AD""",3.75,2.452888,"""Normal""","""Male""",false
"""Adelie""","""AD""",3.8,2.435507,"""Normal""","""Female""",false
"""Adelie""","""AD""",3.25,2.001121,"""Normal""","""Female""",false
"""Adelie""","""AD""",null,null,"""Normal""","""Undetermined""",null
"""Adelie""","""AD""",3.45,2.561456,"""Normal""","""Female""",false
"""Adelie""","""AD""",3.65,2.36324,"""Normal""","""Male""",false
"""Adelie""","""AD""",3.625,2.39557,"""Normal""","""Female""",false
"""Adelie""","""AD""",4.675,3.042352,"""Heavy""","""Male""",true


## Assigning Literal Constants vs Column Expressions
In `mutate()`, column references are explicitly designated via `c.column_name`. Values that are not column expressions are treated as literal constants:
- Writing `status="active"` assigns the literal string value `"active"`.
- Writing `source=c.original` explicitly copies or aliases the column `original`.

Because `c.` distinguishes column references from literals, assigning string constants is completely natural and requires no nested quotes or escaping.


In [27]:
df = pt.DataFrame({"original": [100, 200], "val": [1, 2]})

# Explicit c column assignment copies or aliases a column
print("Copied column 'original':\n", df.mutate(source=c.original))


Copied column 'original':
 shape: (2, 3)
┌──────────┬─────┬────────┐
│ original ┆ val ┆ source │
│ ---      ┆ --- ┆ ---    │
│ i64      ┆ i64 ┆ i64    │
╞══════════╪═════╪════════╡
│ 100      ┆ 1   ┆ 100    │
│ 200      ┆ 2   ┆ 200    │
└──────────┴─────┴────────┘


## Conditional Branching: `if_else` & `case_when`

Pytae provides vectorized conditional branching directly in Python:
- `if_else(condition, if_true, if_false)`: Binary branching. Supports literals or picking between columns!
- `case_when((cond1, val1), (cond2, val2), ..., default=val)`: Multi-tier priority branching with clean tuples.


In [28]:
df_cond = penguins.mutate(size=if_else(c.body_mass_g > 4000, "Heavy", "Normal"))
df_cond.select(c.species, c.body_mass_g, c.size).head(5)


species,body_mass_g,size
str,f64,str
"""Adelie""",3750.0,"""Normal"""
"""Adelie""",3800.0,"""Normal"""
"""Adelie""",3250.0,"""Normal"""
"""Adelie""",null,"""Normal"""
"""Adelie""",3450.0,"""Normal"""


In [29]:
# Dynamic column selection with if_else()
df_col_select = penguins.mutate(metric=if_else(c.island == "Biscoe", c.flipper_length_mm, c.bill_length_mm))
df_col_select.select(c.species, c.island, c.flipper_length_mm, c.bill_length_mm, c.metric).head(5)


species,island,flipper_length_mm,bill_length_mm,metric
str,str,f64,f64,f64
"""Adelie""","""Torgersen""",181.0,39.1,39.1
"""Adelie""","""Torgersen""",186.0,39.5,39.5
"""Adelie""","""Torgersen""",195.0,40.3,40.3
"""Adelie""","""Torgersen""",null,null,null
"""Adelie""","""Torgersen""",193.0,36.7,36.7


In [30]:
# Multi-tier branching with case_when()
df_tier = penguins.mutate(
    tier=case_when(
        (c.body_mass_g > 5000, "Super-Heavy"),
        (c.body_mass_g > 3500, "Medium"),
        default="Light",
    )
)
df_tier.select(c.species, c.body_mass_g, c.tier).head(6)


species,body_mass_g,tier
str,f64,str
"""Adelie""",3750.0,"""Medium"""
"""Adelie""",3800.0,"""Medium"""
"""Adelie""",3250.0,"""Light"""
"""Adelie""",null,"""Light"""
"""Adelie""",3450.0,"""Light"""
"""Adelie""",3650.0,"""Medium"""


## Referencing External Python Variables (`@var`)

Prefix local Python variables with `@` to inject them directly into formula expressions without f-strings or manual escaping.
Unquoted identifiers without `@` always resolve to DataFrame column names, preventing silent shadowing bugs.

In [31]:
cutoff = 4500.0
target_island = "Dream"

(
    penguins
    .mutate(is_target_heavy=if_else((c.body_mass_g >= cutoff) & (c.island == target_island), "Yes", "No"))
    .filter(c.is_target_heavy == "Yes")
    .select(c.species, c.island, c.body_mass_g, c.is_target_heavy)
    .head(5)
)


species,island,body_mass_g,is_target_heavy
str,str,f64,str
"""Adelie""","""Dream""",4650.0,"""Yes"""
"""Adelie""","""Dream""",4600.0,"""Yes"""
"""Chinstrap""","""Dream""",4550.0,"""Yes"""
"""Chinstrap""","""Dream""",4800.0,"""Yes"""
"""Chinstrap""","""Dream""",4500.0,"""Yes"""


## Vectorized Value Recoding: `map(col, mapping[, fallback])`

Recode categorical values using dictionary lookups executed natively in the Polars engine with **Zero Data Loss by default**.

In [32]:
# map: recodes mapped values
(
    penguins
    .mutate(species_code=c.species.map({"Adelie": "AD", "Gentoo": "GEN"}))
    .select(c.species, c.species_code)
    .distinct(c.species, c.species_code)
    .head(5)
)


species,species_code
str,str
"""Adelie""","""AD"""
"""Chinstrap""","""Chinstrap"""
"""Gentoo""","""GEN"""


In [33]:
# map with literal fallback: unmapped rows become 'Other'
(
    penguins
    .mutate(species_code=c.species.map({"Adelie": "AD"}, default="Other"))
    .select(c.species, c.species_code)
    .distinct(c.species, c.species_code)
    .head(5)
)


species,species_code
str,str
"""Adelie""","""AD"""
"""Chinstrap""","""Other"""
"""Gentoo""","""Other"""


## Inline Type Casting: `int(_)`, `float(_)`, `str(_)`, `bool(_)`

Cast column types directly in formula expressions using standard Python constructors:

In [34]:
# Cast mass to string
df_types = penguins.mutate(mass_str=c.body_mass_g.cast(str))
print("mass_str dtype:", df_types.schema["mass_str"])


mass_str dtype: String


## Grouped Window Functions: `first`, `last`, `shift`, `cumsum`

Compute group-partitioned window metrics and baselines without collapsing table rows:

In [35]:
# Compute species mean, deviation, and first/last group baselines
(
    penguins
    .by("species")
    .mutate(
        species_mean=c.body_mass_g.mean(),
        mass_dev=c.body_mass_g - c.body_mass_g.mean(),
        first_mass=c.body_mass_g.first(),
        last_mass=c.body_mass_g.last(),
    )
    .select(c.species, c.body_mass_g, c.species_mean, c.mass_dev, c.first_mass)
    .head(6)
)


species,body_mass_g,species_mean,mass_dev,first_mass
str,f64,f64,f64,f64
"""Adelie""",3750.0,3700.662252,49.337748,3750.0
"""Adelie""",3800.0,3700.662252,99.337748,3750.0
"""Adelie""",3250.0,3700.662252,-450.662252,3750.0
"""Adelie""",null,3700.662252,null,3750.0
"""Adelie""",3450.0,3700.662252,-250.662252,3750.0
"""Adelie""",3650.0,3700.662252,-50.662252,3750.0


## Dynamic Column Resolution via `col(@var)`

When writing dynamic functions or reusable pipeline steps, you can pass column names through local variables using `col(@var)` or `col('name')` rather than formatting strings:


In [36]:
target_measure = "body_mass_g"
(
    penguins
    .mutate(normalized_measure=c[target_measure] / 1000.0)
    .select(c.species, c.body_mass_g, c.normalized_measure)
    .head(4)
)


species,body_mass_g,normalized_measure
str,f64,f64
"""Adelie""",3750.0,3.75
"""Adelie""",3800.0,3.8
"""Adelie""",3250.0,3.25
"""Adelie""",null,null


## Boolean Safety in `map()` and Type Compatibility Guards

When mapping values to booleans, `map()` strictly preserves the boolean dtype (`pl.Boolean`), preventing silent string coercion into truthy `'false'` strings:


In [37]:
df_credit = pt.DataFrame({"rating": ["2B", "3C", "4D"]})
res_bool = df_credit.mutate(
    is_investment_grade=c.rating.map({"2B": True, "3C": False}, default=False)
)
print(res_bool)
print(res_bool.dtypes)


shape: (3, 2)
┌────────┬─────────────────────┐
│ rating ┆ is_investment_grade │
│ ---    ┆ ---                 │
│ str    ┆ bool                │
╞════════╪═════════════════════╡
│ 2B     ┆ true                │
│ 3C     ┆ false               │
│ 4D     ┆ false               │
└────────┴─────────────────────┘
[String, Boolean]


### Strict Type Guards Against Mixing Incompatible Types

Mixing booleans with strings or numeric types in `map()` raises an explicit `TypeError` rather than corrupting types:


In [38]:
try:
    df_credit.mutate(out=c.rating.map({"2B": "Senior"}, default=False))
except (TypeError, Exception) as e:
    print("Caught expected error:", type(e).__name__)


## Unbracketed Column Names with Special Characters

Columns containing parentheses, hyphens, or special characters (common in regulatory reporting like IRBA/LGD) work seamlessly even without brackets:


In [39]:
df_irba = pt.DataFrame({
    "irba_unsecured_lgd_(1g00)": [0.25, 0.45, 0.70],
    "post_crm_lgd_(2a)": [0.20, 0.50, 0.60],
})
display(df_irba.mutate(
    delta=c["irba_unsecured_lgd_(1g00)"] - c["post_crm_lgd_(2a)"],
    is_high_risk=c["irba_unsecured_lgd_(1g00)"] > 0.40,
))


irba_unsecured_lgd_(1g00),post_crm_lgd_(2a),delta,is_high_risk
f64,f64,f64,bool
0.25,0.2,0.05,false
0.45,0.5,-0.05,true
0.7,0.6,0.1,true
